## Анализ данных о Нобелевских лауреатах

### **Контекст проекта**
В этом задании мы будем работать с датасетом о лауреатах Нобелевской премии. Ваша задача — распарсить и проанализировать эти данные, ответив на ключевые вопросы о закономерностях вручения премий, демографических характеристиках лауреатов и исторических тенденциях.

За тесты на функции из отдельных файлов ставится часть оценки за домашку. Напишите их для `json_dict_processing.py`, `dataset_helpers.py`, `aggregations.py` и `atribute_manager.py`. То же для функций в конфигах: `process_year`, процессор призов, процессоры лауреатов. Тестовые файлы создаются рядом с модулями, которые они тестируют.

### **Этап 1: Подготовка данных**

**Задача:** На этом этапе вам надо собрать структурированный список словарей с информацией о лауреатах и призах. Приведем начало нужного нам датасета.

``` {code:python}
[{'id': '745',
  'name': 'A. Michael Spence',
  'gender': 'male',
  'birth_year': 1943,
  'country_birth': 'USA',
  'country_now': 'USA',
  'prizes_relevant': [{'prize_amount': 10000000,
    'prize_amount_adjusted': 15547541,
    'award_year': 2001,
    'category_en': 'Economic Sciences',
    'prize_status': 'received'}],
  'type_': 'person'},
 {'id': '102',
  'name': 'Aage N. Bohr',
  'gender': 'male',
  'birth_year': 1922,
  'country_birth': 'Denmark',
  'country_now': 'Denmark',
  'prizes_relevant': [{'prize_amount': 630000,
    'prize_amount_adjusted': 4304697,
    'award_year': 1975,
    'category_en': 'Physics',
    'prize_status': 'received'}],
  'type_': 'person'}]
  ```

Поскольку информация о лауреатах-людях и лауреатах-организациях структурирована по-разному, а также из-за того, что часть полей в json могут отсутствовать, необходимо реализовать функции для процессинга словарей, которые будут достаточно обобщаемыми для обоих типов данных.

**Что нужно сделать:**

На входе информация о конкретном лауреате представляет из себя кучу вложенных друг в друга словарей. Из этой структуры нам надо получить плоский словарь вида {atribute: value}, где atribute --- это один из небольшого множества интересных нам атрибутов. При этом в исходном файле 
1. пути к атрибутам могут иметь разную длину
2. атрибуты могут отсутствовать
3. атрибуты могут нуждаться в дополнительной обработке (например, поле id в исходных словарях записано как строка.)

Поскольку наивная реализация такого обработчика оказывается довольно громоздкой, вам нужно создать файл json_dict_processing.py, в котором реализован высокоуровневый функционал для решения задачи "обработать вложенный json-словарь"

Главное требование к интерфейсу: 
по (вложенному) словарю о лауреате и конфигу (см ниже) надо получить плоский словарь вида {atribute: value}.

Достичь этого можно, например, таким набором функций

* `extract_nested_value(obj, keys)` - извлекает значение из вложенных структур данных по цепочке ключей

* `process_dictionary_with_config(dictionary, config)` - обрабатывает один словарь согласно словару-конфигу (формат конфига см ниже.)

* `process_list_of_dicts_with_config(list_of_dicts, config)` - обрабатывает список словарей, эта функция нужна чтобы обработать список призов.

* `create_processor(config, list_processor=False)` - создает готовый процессор с предзагруженной конфигурацией. В случае, если list_processor == True, внутрь процессора подается process_list_of_dicts_with_config, иначе process_dictionary_with_config.

**Замечание насчет последней функции.**

Планируемое применение этой функции примерно такое:
`laureate_processor = create_processor(PERSON_CONFIG, list_processor=False)`

То есть, create_processor возвращает функцию, которая реализует функционал process_dictionary_with_config но с уже заполненным конфигом. Это повышает читаемость кода и снижает риск перепутать конфиги из-за дублирования кода.


Помимо этого нужно создать файлы-конфиги для данных по призам, лауреатам-людям, и лауреатам-организациям.

КОНФИГ для функций process_dictionary_with_config и process_list_of_dicts_with_config имеет следующий вид
```python
  {'атрибут_который_берем_сырым': ['путь', 'к', 'данным'],  # или
  'атрибут_который_нужно_обработать': (['путь', 'к', 'данным'], функция_обработки)}
  ```
В случае, если атрибут можно взять в итоговый датасет сырым, в values() словаря-конфига будет список path_list.
Если же атрибут еще надо обработать, для него в поле config[attribute] будет tuple вида (path_list, function).

Примеры атрибутов, которые можно взять сырыми: имя, размер приза. Примеры атрибутов, которые нужно обработать: год вручения приза (в сырых данных есть только полная дата, записанная как строка)

После того, как будет создан core.py, создайте конфиги laureates_configs.py и prizes_configs.py

пример использования: 

assert 1 == True

**Валидация**

В вашем случае assert должен взять простой словарик с одним вложенным полем и какой-нибудь конфиг-файл.
Также убедитесь, что при обработке данных собственно по лауреатам в них нет странностей (смотрите ниже) 

**Атрибуты для ПРИЗА (CONFIG_PRIZE, prizes_configs.py)**

- `prize_amount` - сумма приза → `prizeAmount`
- `prize_amount_adjusted` - скорректированная сумма приза → `prizeAmountAdjusted`
- `award_year` - год вручения приза (преобразуется в целое число) → `awardYear`
- `category_en` - категория на английском → `category.en`
- `prize_status` - статус приза → `prizeStatus`

**Атрибуты для ЧЕЛОВЕКА-ЛАУРЕАТА (CONFIG_PERSON, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - имя лауреата → `knownName.en`
- `gender` - пол → `gender`
- `birth_year` - год рождения (извлекается из даты) → `birth.date`
- `country_birth` - страна рождения → `birth.place.country.en`
- `country_now` - текущая страна → `birth.place.countryNow.en`
- `prizes_relevant` - список призов лауреата → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Атрибуты для ОРГАНИЗАЦИИ-ЛАУРЕАТА (CONFIG_ORG, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - название организации → `orgName.en`
- `founded_year` - год основания (извлекается из даты) → `founded.date`
- `country_founded` - страна основания → `founded.place.country.en`
- `country_now` - текущая страна → `founded.place.countryNow.en`
- `prizes_relevant` - список призов организации → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Вспомогательные функции:**

- `process_year(year_string)` - извлекает год из строки даты (формат "YYYY-MM-DD"). Функция находится в файле laureates_configs.py
- `prize_processor()` - создает процессор для обработки списков призов. Функция находится в файле prizes_configs.py
- `person_processor()` - создает процессор для данных о людях-лауреатах. Функция находится в файле laureates_configs.py
- `org_processor()` - создает процессор для данных об организациях-лауреатах. Функция находится в файле laureates_configs.py

In [ ]:
import requests
from laureates_configs import person_processor, org_processor
from dataset_helpers import (
    count_missing_values,
    calculate_missing_ratios,
    filter_data,
    group_by_attributes,
    apply_aggregation
)
from aggregations import (
    get_top_n,
    calculate_mean,
    calculate_median,
    calculate_max,
    calculate_min
)
from atribute_manager import add_attribute

In [2]:
URL_LAUREATES = f'https://api.nobelprize.org/2.1/laureates?limit=1200'

headers = {'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'}

laureates_response = requests.get(URL_LAUREATES, headers=headers)
laureates = laureates_response.json()
laureates

{'laureates': [{'id': '745',
   'knownName': {'en': 'A. Michael Spence', 'se': 'A. Michael Spence'},
   'givenName': {'en': 'A. Michael', 'se': 'A. Michael'},
   'familyName': {'en': 'Spence', 'se': 'Spence'},
   'fullName': {'en': 'A. Michael Spence', 'se': 'A. Michael Spence'},
   'fileName': 'spence',
   'gender': 'male',
   'birth': {'date': '1943-11-07',
    'year': '1943',
    'place': {'city': {'en': 'Montclair, NJ',
      'no': 'Montclair, NJ',
      'se': 'Montclair, NJ'},
     'country': {'en': 'USA', 'no': 'USA', 'se': 'USA'},
     'cityNow': {'en': 'Montclair, NJ',
      'no': 'Montclair, NJ',
      'se': 'Montclair, NJ',
      'sameAs': ['https://www.wikidata.org/wiki/Q678437',
       'https://www.wikipedia.org/wiki/Montclair,_New_Jersey'],
      'latitude': '40.825930',
      'longitude': '-74.209030'},
     'countryNow': {'en': 'USA',
      'no': 'USA',
      'se': 'USA',
      'sameAs': ['https://www.wikidata.org/wiki/Q30'],
      'latitude': '39.828175',
      'longitu

In [3]:
def get_laureate_data(laureate: dict) -> dict:
    """Принимает словарь с исходными данными лауреата.
    Определяет тип лауреата и обрабатывает данные соответствующим процессором.
    Возвращает обработанный словарь с добавленным полем type_."""
    if 'knownName' in laureate:
        processor = person_processor()
        processed_data = processor(laureate)
        processed_data['type_'] = 'person'
    elif 'orgName' in laureate:
        processor = org_processor()
        processed_data = processor(laureate)
        processed_data['type_'] = 'org'
    else:
        raise ValueError('Не удалось определить тип лауреата')
    return processed_data


laureates_data = []

for laureate in laureates['laureates']:
    laureates_data.append(get_laureate_data(laureate))



### **Этап 2: EDA лауреатов**

EDA или Exploratory Data Analysis - первый шаг при работе с любыми данными.

Чтобы начать работать с этой секцией, проведите проведите чисто технический анализ собранных данных.

1. Сколько всего записей?
2. Все ли значения полей заполнены?
3. Если нет, сколько пропусков в каждом из полей? Выведите ответ на этот вопрос в формате `{'field_name': missing_values}`. А какая доля пропусков?
5. Каковы максимальные и минимальные значения поля id и каким годам награждения они соответствуют? Есть ли записи с повторяющимися id? Есть ли пары айди, которые в отсортированном массиве айдишников различаются более, чем на один? 

При реализации ответов на эти вопросы, пользуйтесь теми же принципами, что описаны выше. Можете начать с плохого хакерского кода, однако в таком случае в маркдауне пропишите, как будете его обобщать и разносить по функциям и модулям. (об этом ниже)

В основном EDA пользуйтесь аккуратным кодом, разбитым на функции и файлы.

### 1. Сколько всего записей?

In [4]:
# Общее количество записей в наборе данных.
print(len(laureates_data))

1028


### 2. Все ли значения полей заполнены?

In [5]:
# Разделяем лауреатов на физических лиц и организации по полю type_,
# т.к. они имеют разные наборы атрибутов.

person_laureates = [
    laureate for laureate in laureates_data if laureate['type_'] == 'person'
]

org_laureates = [
    laureate for laureate in laureates_data if laureate['type_'] == 'org'
]

In [6]:
# Создадим функцию для читаемого вывода количества и доли пропусков
def print_missing_statistics(data: list[dict], title: str) -> None:
    """Принимает список словарей и название группы.
    Выводит количество и долю пропусков по каждому полю."""
    missing_counts = count_missing_values(data)
    missing_ratios = calculate_missing_ratios(data)
    print(title, ':', sep='')
    for field in missing_counts:
        print(
            f'Поле {field} содержит {missing_counts[field]} пропусков, '
            f'доля составила {round(missing_ratios[field] * 100, 2)}%'
        )
    print()

### Количество и доля пропусков по каждому полю для физических лиц и для организаций.

In [7]:
# Создаем функции count_missing_values и calculate_missing_ratios
# в модуле dataset_helpers.py для подсчёта количества и доли пропусков
# по каждому полю набора данных.
print_missing_statistics(person_laureates, 'Физические лица')
print_missing_statistics(org_laureates, 'Организации')

Физические лица:
Поле id содержит 0 пропусков, доля составила 0.0%
Поле name содержит 0 пропусков, доля составила 0.0%
Поле gender содержит 0 пропусков, доля составила 0.0%
Поле birth_year содержит 0 пропусков, доля составила 0.0%
Поле country_birth содержит 3 пропусков, доля составила 0.3%
Поле country_now содержит 3 пропусков, доля составила 0.3%
Поле prizes_relevant содержит 0 пропусков, доля составила 0.0%
Поле type_ содержит 0 пропусков, доля составила 0.0%

Организации:
Поле id содержит 0 пропусков, доля составила 0.0%
Поле name содержит 0 пропусков, доля составила 0.0%
Поле founded_year содержит 1 пропусков, доля составила 3.57%
Поле country_founded содержит 4 пропусков, доля составила 14.29%
Поле country_now содержит 4 пропусков, доля составила 14.29%
Поле prizes_relevant содержит 0 пропусков, доля составила 0.0%
Поле type_ содержит 0 пропусков, доля составила 0.0%



Вывод: у физических лиц пропусков почти нет, максимум 0,3%.

У организаций доля пропусков выше - до 14,29%. Больше всего пропусков в полях, связанных со странами.

### 3. Если нет, сколько пропусков в каждом из полей? Выведите ответ на этот вопрос в формате `{'field_name': missing_values}`. А какая доля пропусков?

In [8]:
# Выведем количество и доли в требуемом формате
print('Физические лица:')
print(count_missing_values(person_laureates))
print(calculate_missing_ratios(person_laureates))
print()

print('Организации:')
print(count_missing_values(org_laureates))
print(calculate_missing_ratios(org_laureates))

Физические лица:
{'id': 0, 'name': 0, 'gender': 0, 'birth_year': 0, 'country_birth': 3, 'country_now': 3, 'prizes_relevant': 0, 'type_': 0}
{'id': 0.0, 'name': 0.0, 'gender': 0.0, 'birth_year': 0.0, 'country_birth': 0.003, 'country_now': 0.003, 'prizes_relevant': 0.0, 'type_': 0.0}

Организации:
{'id': 0, 'name': 0, 'founded_year': 1, 'country_founded': 4, 'country_now': 4, 'prizes_relevant': 0, 'type_': 0}
{'id': 0.0, 'name': 0.0, 'founded_year': 0.0357, 'country_founded': 0.1429, 'country_now': 0.1429, 'prizes_relevant': 0.0, 'type_': 0.0}


### 4. Каковы максимальные и минимальные значения поля id и каким годам награждения они соответствуют? Есть ли записи с повторяющимися id? Есть ли пары айди, которые в отсортированном массиве айдишников различаются более, чем на один? 


In [9]:
def analyze_ids(data: list[dict]) -> None:
    """Принимает список словарей с данными о лауреатах.
    Выводит минимальный и максимальный ID, соответствующие годы
    награждения,
    наличие повторяющихся ID и разрывов в их последовательности."""
    # Определяем минимальный и максимальный ID.
    ids = [int(laureate['id']) for laureate in data]
    max_id = max(ids)
    min_id = min(ids)

    # Находим лауреатов и соответствующие годы награждения.
    max_laureate = [
        laureate for laureate in data if laureate['id'] == str(max_id)
    ][0]
    min_laureate = [
        laureate for laureate in data if laureate['id'] == str(min_id)
    ][0]

    years_max = [
        prize['award_year'] for prize in max_laureate['prizes_relevant']
    ]
    years_min = [
        prize['award_year'] for prize in min_laureate['prizes_relevant']
    ]

    # Выводим крайние ID и соответствующие годы награждения.
    print(f'Максимальный id составил: {max_id}')
    print(f'Годы награждения: {years_max}')
    print()
    print(f'Минимальный id составил: {min_id}')
    print(f'Годы награждения: {years_min}')
    print()

    # Проверяем наличие повторяющихся ID.
    if len(set(ids)) < len(ids):
        print('Есть записи с повторяющимися id')
    else:
        print('Нет записей с повторяющимися id')
    print()

    # Проверяем наличие разрывов в последовательности ID.
    ids = sorted(ids)
    differences = [ids[i + 1] - ids[i] for i in range(len(ids) - 1)]
    if max(differences, default=0) > 1:
        print(
            'Есть пары айди, '
            'которые в отсортированном массиве айдишников различаются более, '
            'чем на один'
        )
    else:
        print(
            'Нет пар айди, '
            'которые в отсортированном массиве айдишников различаются более, '
            'чем на один'
        )

In [10]:
analyze_ids(laureates_data)

Максимальный id составил: 1070
Годы награждения: [2026]

Минимальный id составил: 1
Годы награждения: [1901]

Нет записей с повторяющимися id

Есть пары айди, которые в отсортированном массиве айдишников различаются более, чем на один


Вывод: значения ID лежат в диапазоне от 1 до 1070 и соответствуют годам награждения 1901 и 2026.

Повторяющиеся ID отсутствуют, но в их последовательности имеются пропуски.


Поскольку это курс по Python, а не по статистике или визуализации данных, мы не задаем жесткие рамки того, что вы должны проанализировать.
Однако это отличная возможность представить себя в роли разработчика базы данных или, например, библиотеки pandas.

В коде нужно соблюдать следующие правила:

1. должны быть реализованы  несколько способов работы с данными (среднее/медианное/максимальное значение метрики, количество примеров, вывод топ-N объектов по значению метрики).
2. анализ выбранных вами тем должен включать в себя анализ метрик в тотале, по категориям и в динамике по годам или десятилетиям.
3. проанализировать стоит 2-3 темы. Вы можете использовать темы ниже или взять свои.
2. вывод ноутбука должен быть понятен обывателю. Это может быть визуализация в matplotlib/seaborn. Сопровождайте вывод markdown-ячейками с вашими  мыслями, пояснениями и инсайтами, которые удалось извлечь.
3. Нельзя использовать pandas и numpy, но можно вдохновляться их интерфейсом и функционалом при дизайне ваших функций.


Из пункта 2 следует, что вам придется работать не только с атрибутами лауреатов, но и самих призов.
Информацию о призах нужно доставать и записывать как атрибут уже лауреата. Функции, работающие с призами (например, факт того, что лауреат отказался от какого-то из призов) можно реализовать в ноутбуке, но если вы создаете более общие вспомогательные функции, выносите их в отдельные модули.

Мы рекомендуем следующую структуру кода:

* `dataset_helpers.py` в котором реализованы фильтрация списка словарей (плоских) по условиям (аналог sql-ного where), а также аналог sql-ного групбая. А также функция для применения агрегаций к списку словарей
* `aggregations.py` в котором реализованы функции расчета среднего/медианы/других эвристик и мб вывод топ n-значений по какой-то метрике.
* `atribute_manager.py` в котором реализован переиспользуемый интерфейс для добавления (и мб удаления) полей.

Замечание насчет group-by. 

На игрушесном примере она может работать примерно так:
```
    Входные данные: [
        {'name': 'Alice', 'country': 'USA', 'age': 25},
        {'name': 'Bob', 'country': 'UK', 'age': 30},
        {'name': 'Charlie', 'country': 'USA', 'age': 25}
    ]
    
    group_by_attributes(data, ['country', 'age']) → {
        ('USA', 25): [
            {'name': 'Alice', 'country': 'USA', 'age': 25},
            {'name': 'Charlie', 'country': 'USA', 'age': 25}
        ],
        ('UK', 30): [
            {'name': 'Bob', 'country': 'UK', 'age': 30}
        ]
    }
```

Замечание насчет agregations.py
1. Агрегации соответственно должны применяться к спискам вида [
    {'name': 'Alice', 'country': 'USA', 'age': 25},
    {'name': 'Charlie', 'country': 'USA', 'age': 25}
]
2. Пользуйтесь модулем statistics из стандартной библиотеки питона.


Эта структура одна из возможных для организации вашего кода, главное, чтобы эти достаточно общие функции блыи переиспользованы

----------

При анализе данных вам понадобятся более конкретные функции, например filter_by_num_prizes или output_metric_by_category. Их пишите прямо в ноутбуке и сопровождайте коментами.

Ниже мы приводим некоторые примеры вопросов, которые может быть интересно задать к данным, но если у вас есть желание, можете поисследовать датасет с любой интересной вам стороны!


**2.1. Топ-статистика по странам**
- Определите топ-5 стран по общему количеству лауреатов
- Постройте аналогичные рейтинги отдельно по категориям (физика, химия, медицина и т.д.)

**2.2. Возраст при награждении**
- Рассчитать средний возраст лауреатов на момент получения **первого** приза. Рассмотрите только лауреатов-людей.
- Исследуйте, как средний возраст получения первого приза различается по категориям
- Различаются ли средний и медианный возрасты? 

**2.3. Гендерное распределение**
- Как менялось количество женщин-лауреатов по десятилетиям?
- Сравните распределение лауреаток по категориям. Выведите значения суммарно за все годы и по декадам.
- В какой из категорий больше всего женщин лауреаток изначально и в последние пару десятилетий?

**2.4. Миграции лауреатов**
- У скольки лауреатов исходная страна и страна на момент вручения нобелевской премии различаются? Всегда ли это различие объясняется миграцией? Добавьте вспомогательное поле 'country_of_origin_adjusted_for_migration', чтобы различие между этим полем и страной на момент вручения объяснялось миграцией.
- Выявите страны с наибольшей долей "оттекших" и "притекших" лауреатов.
- Есть ли оттоки среди организаций?

**2.5. Многократные лауреаты**
- Найдите лауреатов, получившие больше одного приза. Кто получил больше 2 раз? 
- Совпадает ли категория второго приза с первой?  
- Сколько в среднем времени прошло между получением первой и второй премии?

**2.6 Отказы от премий**
- Определите, в каких категориях чаще отказывались от премий?
- Есть ли отказы среди организаций?

In [11]:
def has_prize_category(laureate: dict, category: str) -> bool:
    """Принимает словарь с данными о лауреате и категорию премии.
    Возвращает True если есть премии в указанной категории,
    иначе False."""
    return any(
        prize['category_en'] == category
        for prize in laureate['prizes_relevant']
    )

**2.1. Топ-статистика по странам**
- Определите топ-5 стран по общему количеству лауреатов



In [12]:
def get_first_award_year(
    laureate: dict, category: str | None = None
) -> int | None:
    """Возвращает год первого награждения лауреата
    по указанной категории или среди всех категорий.
    При отсутствии годов награждения возвращает None."""
    years = []
    for prize in laureate['prizes_relevant']:
        if prize['award_year'] is not None:
            if category is None or prize['category_en'] == category:
                years.append(prize['award_year'])
    if years:
        return min(years)
    return None


def calculate_award_decade_by_category(
    laureate: dict, category: str | None = None
) -> int | None:
    """Принимает словарь с данными о лауреате.
    Возвращает десятилетие первого награждения
    или None при отсутствии годов награждения."""

    min_year = get_first_award_year(laureate, category)
    if min_year is not None:
        decade = min_year - min_year % 10
        return decade
    return None

In [ ]:
def get_top_countries(
    person_laureates: list[dict], n: int, category: str | None = None
) -> dict:
    """Возвращает топ-N стран по количеству родившихся в них
    лауреатов в целом или по категории."""
    filter_person_laureates = filter_data(
        person_laureates,
        lambda laureate: (
            laureate['country_birth'] is not None
            and (category is None or has_prize_category(laureate, category))
        )
    )
    # Если category это None, то условие выполняется, если не None,
    # то выоплнится только если втрое выполняется

    laureates_by_country = group_by_attributes(
        filter_person_laureates, ['country_birth']
    )
    laureate_counts_by_country = apply_aggregation(laureates_by_country, len)
    return get_top_n(laureate_counts_by_country, n)

In [14]:
top_5_countries = get_top_countries(person_laureates, 5)
print(top_5_countries)

{('USA',): 299, ('United Kingdom',): 94, ('Germany',): 82, ('France',): 61, ('Japan',): 31}


Вывод: лидером с больщим отрывом по количеству лауреатов по стране рождения являются США (299 человек). Второе и третье места занимают Великобритания (94) и Германия (82). США имеют значительный отрыв от остальных стран.

- Постройте аналогичные рейтинги отдельно по категориям (физика, химия, медицина и т.д.)

In [ ]:
categories = [
    'Physics',
    'Chemistry',
    'Physiology or Medicine',
    'Literature',
    'Peace',
    'Economic Sciences'
]
for category in categories:
    top_5_countries = get_top_countries(person_laureates, 5, category)
    print(f'Категория: {category}')
    print(top_5_countries)
    print()

Категория: Physics
{('USA',): 71, ('United Kingdom',): 25, ('Germany',): 24, ('Japan',): 12, ('France',): 11}

Категория: Chemistry
{('USA',): 59, ('United Kingdom',): 26, ('Germany',): 24, ('France',): 12, ('Japan',): 9}

Категория: Physiology or Medicine
{('USA',): 85, ('United Kingdom',): 25, ('Germany',): 22, ('France',): 12, ('Sweden',): 8}

Категория: Literature
{('France',): 12, ('USA',): 10, ('Sweden',): 7, ('United Kingdom',): 6, ('Spain',): 5}

Категория: Peace
{('USA',): 19, ('France',): 9, ('Germany',): 6, ('Sweden',): 5, ('United Kingdom',): 5}

Категория: Economic Sciences
{('USA',): 56, ('United Kingdom',): 7, ('Canada',): 5, ('France',): 5, ('the Netherlands',): 4}



Вывод: США лидирует по количеству премий по всем категориям, кроме литературы, где лучшая - Франция. Великобритания занимает второе место в 4 из 6 категориях.

- Исследуем изменение топ-5 стран по количеству лауреатов в динамике по десятилетиям.

In [16]:
person_laureates_with_decade = add_attribute(
    person_laureates, 'first_award_decade', calculate_award_decade_by_category
)

laureates_by_decade = group_by_attributes(
    person_laureates_with_decade, ['first_award_decade']
)
laureates_by_decade = dict(
    sorted(laureates_by_decade.items(), key=lambda x: x[0])
)
print(
    'Топ 5 стран по количеству родившихся лауреатов, '
    'впервые получивших Нобелевскую премию в'
)
for decade in laureates_by_decade:
    print(
        f'{decade[0]}-е: {get_top_countries(laureates_by_decade[decade], 5)}'
    )
    print()

Топ 5 стран по количеству родившихся лауреатов, впервые получивших Нобелевскую премию в
1900-е: {('France',): 9, ('Prussia',): 6, ('Switzerland',): 4, ('the Netherlands',): 3, ('Russian Empire',): 3}

1910-е: {('France',): 6, ('Prussia',): 4, ('Germany',): 4, ('Sweden',): 3, ('USA',): 3}

1920-е: {('Germany',): 10, ('France',): 8, ('United Kingdom',): 7, ('USA',): 4, ('Denmark',): 4}

1930-е: {('USA',): 14, ('United Kingdom',): 10, ('Germany',): 9, ('Austria-Hungary',): 3, ('France',): 3}

1940-е: {('USA',): 13, ('Austria-Hungary',): 4, ('Germany',): 4, ('United Kingdom',): 3, ('Denmark',): 2}

1950-е: {('USA',): 21, ('United Kingdom',): 10, ('Germany',): 9, ('Russia',): 5, ('France',): 3}

1960-е: {('USA',): 20, ('Germany',): 11, ('United Kingdom',): 7, ('France',): 5, ('Australia',): 3}

1970-е: {('USA',): 32, ('United Kingdom',): 13, ('Germany',): 8, ('Sweden',): 6, ('Russian Empire',): 5}

1980-е: {('USA',): 31, ('Germany',): 7, ('Sweden',): 5, ('France',): 4, ('United Kingdom',): 

Вывод: в первые два десятилетия по количеству лауреатов первой была Франция, в 1920-е годы первой стала Германия. Начиная с 1930-х годов первое место во всех десятилетиях занимают США, причем с 1970-х годов с большим отрывом от остальных стран. Наибольшее количество лауреатов из США наблюдается в 2000-е годы (52 человека). Также интересно, что с 2000-х годов в топ-5 стран постоянно входит Япония.

**2.2. Возраст при награждении**
- Различаются ли средний и медианный возрасты?
- Рассчитать средний возраст лауреатов на момент получения **первого** приза. Рассмотрите только лауреатов-людей.

In [17]:
def calculate_age_at_first_award(laureate: dict) -> int | None:
    """Принимает словарь с данными о лауреате.
    Возвращает возраст при первом награждении
    или None при отсутствии необходимых данных."""
    min_year = get_first_award_year(laureate)

    if laureate['birth_year'] is not None and min_year is not None:
        return min_year - laureate['birth_year']
    else:
        return None


person_laureates_with_age = add_attribute(
    person_laureates, 'age_at_first_award', calculate_age_at_first_award
)

mean_age = calculate_mean(person_laureates_with_age, 'age_at_first_award')

median_age = calculate_median(person_laureates_with_age, 'age_at_first_award')

print(
    'Средний возраст лауреатов на момент '
    f'получения певой премии составил: {mean_age}'
)
print(
    'Медианный возраст лауреатов на момент '
    f'получения певой премии составил: {median_age}'
)

Средний возраст лауреатов на момент получения певой премии составил: 60.606
Медианный возраст лауреатов на момент получения певой премии составил: 61.0


Вывод: средний возраст получения первой Нобелевской премии составил 60,6 год, медианный же составил 61 год. Присутствует незначительное расхождение в среднем и медианном возрасте, распределение смещено вправо (основная масса наблюдений).

- Исследуйте, как средний возраст получения первого приза различается по категориям

In [ ]:
for category in categories:
    filter_person_laureates = filter_data(
        person_laureates_with_age,
        lambda laureate: has_prize_category(laureate, category)
    )
    mean_age = calculate_mean(filter_person_laureates, 'age_at_first_award')
    median_age = calculate_median(
        filter_person_laureates, 'age_at_first_award'
    )

    print(f'Категория: {category}')
    print(
        'Средний возраст лауреатов на момент '
        f'получения певой премии составил: {round(mean_age, 4)}'
    )
    print(
        'Медианный возраст лауреатов на момент '
        f'получения певой премии составил: {median_age}'
    )
    print()

Категория: Physics
Средний возраст лауреатов на момент получения певой премии составил: 57.7913
Медианный возраст лауреатов на момент получения певой премии составил: 56.0

Категория: Chemistry
Средний возраст лауреатов на момент получения певой премии составил: 59.315
Медианный возраст лауреатов на момент получения певой премии составил: 58.0

Категория: Physiology or Medicine
Средний возраст лауреатов на момент получения певой премии составил: 59.0213
Медианный возраст лауреатов на момент получения певой премии составил: 58

Категория: Literature
Средний возраст лауреатов на момент получения певой премии составил: 65.0407
Медианный возраст лауреатов на момент получения певой премии составил: 67

Категория: Peace
Средний возраст лауреатов на момент получения певой премии составил: 60.9027
Медианный возраст лауреатов на момент получения певой премии составил: 62

Категория: Economic Sciences
Средний возраст лауреатов на момент получения певой премии составил: 67.2079
Медианный возраст 

Вывод: самый низкий средний возраст получения первой Нобелевской премии наблюдается у лауреатов по физике (57,8 года), а самый высокий в экономике (67,2 года). В физике, химиии, медицине средний возраст превышает мелианный, а в литературе и премии мира наблюдается обратная ситуация. В экономике средний и медианный возраст практически совпадают.

- Исследуем минимальный, средний, медианный и максимальный возраст лауреатов на момент получения первой премии по десятилетиям.

In [ ]:
person_laureates_with_age = add_attribute(
    person_laureates_with_decade,
    'age_at_first_award',
    calculate_age_at_first_award
)
laureates_by_decade = group_by_attributes(
    person_laureates_with_age, ['first_award_decade']
)

laureates_by_decade = dict(
    sorted(laureates_by_decade.items(), key=lambda x: x[0])
)

print(
    'Минимальный, средний, медианный и максимальный возраст лауреатов '
    'на момент получения первой премии в'
)
for decade in laureates_by_decade:
    min_age = calculate_min(laureates_by_decade[decade], 'age_at_first_award')

    mean_age = round(
        calculate_mean(laureates_by_decade[decade], 'age_at_first_award'), 2
    )

    median_age = round(
        calculate_median(laureates_by_decade[decade], 'age_at_first_award'), 2
    )

    max_age = calculate_max(laureates_by_decade[decade], 'age_at_first_award')

    print(
        f'{decade[0]}-e годы: минимальный - {min_age}, '
        f'средний - {mean_age}, медианный - {median_age}, '
        f'максимальный - {max_age}'
    )
    print()

Минимальный, средний, медианный и максимальный возраст лауреатов на момент получения первой премии в
1900-e годы: минимальный - 35, средний - 57.84, медианный - 58.0, максимальный - 85

1910-e годы: минимальный - 25, средний - 53.08, медианный - 50, максимальный - 80

1920-e годы: минимальный - 32, средний - 54.98, медианный - 56.0, максимальный - 86

1930-e годы: минимальный - 31, средний - 51.62, медианный - 49, максимальный - 75

1940-e годы: минимальный - 39, средний - 58.02, медианный - 56.0, максимальный - 81

1950-e годы: минимальный - 31, средний - 54.32, медианный - 54, максимальный - 79

1960-e годы: минимальный - 32, средний - 56.45, медианный - 57.0, максимальный - 87

1970-e годы: минимальный - 32, средний - 59.34, медианный - 59.0, максимальный - 87

1980-e годы: минимальный - 37, средний - 60.54, медианный - 60, максимальный - 83

1990-e годы: минимальный - 33, средний - 62.14, медианный - 62, максимальный - 87

2000-e годы: минимальный - 40, средний - 65.87, медианный -

Вывод: с 1950-х годов средний и мелианный возраст лауреатов постепенно увеличивается. Если в 1950-е годы средний возраст составлял 54 года, то в 2020-е — уже 69 лет. Максимальный возраст также растет, а по минимальному возрасту явной тенденции нет. Удивительно, что в 2010-х годах кто-то получил премию в 17 лет. Также сильно выделяется номинант в 1910-х годах, получивший премию в 25 лет. Удивительны и номинанты, получившие премии в 96 и 97 лет.

**2.3. Гендерное распределение**
- Как менялось количество женщин-лауреатов по десятилетиям?

In [ ]:
def count_female_laureates_by_decade(
    person_laureates: list[dict], category: str | None = None
) -> dict:
    """Возвращает количество лауреатов по десятилетиям
    в целом или по категории."""
    female_laureates = filter_data(
        person_laureates,
        lambda laureate: (
            laureate['gender'] == 'female'
            and (category is None or has_prize_category(laureate, category))
        )
    )

    female_laureates_with_decade = add_attribute(
        female_laureates,
        'first_award_decade',
        lambda laureate: calculate_award_decade_by_category(
            laureate, category
        )
    )

    female_laureates_by_decade = group_by_attributes(
        female_laureates_with_decade, ['first_award_decade']
    )

    female_counts_by_decade = apply_aggregation(
        female_laureates_by_decade, len
    )
    return dict(sorted(female_counts_by_decade.items(), key=lambda x: x[0]))


female_counts_by_decade = count_female_laureates_by_decade(person_laureates)
for dec in female_counts_by_decade:
    print(
        f'В {dec[0]}-е годы впервые Нобелевскую премию получили '
        f'{female_counts_by_decade[dec]} женщин'
    )
    print()

В 1900-е годы впервые Нобелевскую премию получили 3 женщин

В 1920-е годы впервые Нобелевскую премию получили 2 женщин

В 1930-е годы впервые Нобелевскую премию получили 3 женщин

В 1940-е годы впервые Нобелевскую премию получили 3 женщин

В 1960-е годы впервые Нобелевскую премию получили 3 женщин

В 1970-е годы впервые Нобелевскую премию получили 4 женщин

В 1980-е годы впервые Нобелевскую премию получили 4 женщин

В 1990-е годы впервые Нобелевскую премию получили 7 женщин

В 2000-е годы впервые Нобелевскую премию получили 11 женщин

В 2010-е годы впервые Нобелевскую премию получили 13 женщин

В 2020-е годы впервые Нобелевскую премию получили 16 женщин



Вывод: с 1990-х годов наблюжается устойчивое увеличение количества впервые получивших Нобелевскую премию женщин. Если с 1900 по 1960-е года Нобелевкую премию получали всего по 3 женщины за десятилетие, а в 20 веке за десятилетие Нобелевкую премию получало не более 7 женщин, то в 2020-е Нобелевскую премию уже получили 16 женщин, несмотря на то, что десятилетие еще не завершилось.

Также интересно, что за 20 век Нобелевскую премию в сумме получило меньше женщин, чем за первую четверть 21 века.

- Сравните распределение лауреаток по категориям. Выведите значения суммарно за все годы и по декадам.
- В какой из категорий больше всего женщин лауреаток изначально и в последние пару десятилетий?
за все годы и по декадам

In [21]:
# Словари для статистики по первым 2 десятилетиям и по последним
early_counts = {}
recent_counts = {}

for category in categories:
    female_counts_by_decade = count_female_laureates_by_decade(
        person_laureates, category
    )

    # Агрегация по 2 десятилетиям
    early_counts[category] = sum(
        count
        for decade, count in female_counts_by_decade.items()
        if decade[0] < 1920
    )

    recent_counts[category] = sum(
        count
        for decade, count in female_counts_by_decade.items()
        if decade[0] >= 2010
    )

    print(f'Категория: {category}')
    print(f'Всего лауреаток: {sum(female_counts_by_decade.values())}')
    print(female_counts_by_decade)
    print()

print(
    'Количество женщин, '
    'впервые получивших Нобелевскую премию в конкретной категории '
    'за первые 2 десятилетия:'
)
print(get_top_n(early_counts, 6))
print()

print(
    'Количество женщин, '
    'впервые получивших Нобелевскую премию в конкретной категории '
    'за последние 2 десятилетия:'
)
print(get_top_n(recent_counts, 6))

Категория: Physics
Всего лауреаток: 5
{(1900,): 1, (1960,): 1, (2010,): 1, (2020,): 2}

Категория: Chemistry
Всего лауреаток: 8
{(1910,): 1, (1930,): 1, (1960,): 1, (2000,): 1, (2010,): 1, (2020,): 3}

Категория: Physiology or Medicine
Всего лауреаток: 14
{(1940,): 1, (1970,): 1, (1980,): 3, (1990,): 1, (2000,): 4, (2010,): 2, (2020,): 2}

Категория: Literature
Всего лауреаток: 19
{(1900,): 1, (1920,): 2, (1930,): 1, (1940,): 1, (1960,): 1, (1990,): 3, (2000,): 3, (2010,): 3, (2020,): 4}

Категория: Peace
Всего лауреаток: 21
{(1900,): 1, (1930,): 1, (1940,): 1, (1970,): 3, (1980,): 1, (1990,): 3, (2000,): 2, (2010,): 5, (2020,): 4}

Категория: Economic Sciences
Всего лауреаток: 3
{(2000,): 1, (2010,): 1, (2020,): 1}

Количество женщин, впервые получивших Нобелевскую премию в конкретной категории за первые 2 десятилетия:
{'Physics': 1, 'Chemistry': 1, 'Literature': 1, 'Peace': 1, 'Physiology or Medicine': 0, 'Economic Sciences': 0}

Количество женщин, впервые получивших Нобелевскую прем

Вывод: за всю историю Нобелевской премии наибольшее количество женщин-лауреатов наблюдается в категориях премии мира (21) и литературы (19), наименьшее количество женщин получили премии по экономике (3) и по физике (5).

В первые два десятилетия по одной женщине получили премии по физике, химии, литературе и премию мира, по остальным дисциплинам по 0. За последние два десятилетия бнаибольшее количество премий мира (9 лауреаток) и премий по литературе (7 лауреаток). Также есть хотя бы 2 премии по каждой категории.

---------

## Бонусное задание на +2 балла


### Лингвистический анализ мотиваций**

**5.1. Анализ текстов мотиваций**
- Соберите корпус текстов мотиваций вручения премий. Для этого измените конфиг обработки призов.
- Выделите самые частые топ-50 слов. Какие из этих слов самые частые в принципе в английском языке (такие, как for, the и другие), а какие кажутся важными именно в контексте вручения Нобелевской премии (например, 'chemistry')?
- Самые частые слова, которые также являются частыми во всем английском языке занесите в отдельный конфиг-файл под названием `stopwords.py`. Вычистите их из текстов. Какие слова являются самыми частыми теперь? Сравните множества топ-10 самых частых слов по разным категориям


- Если есть желание, посчитайте TF-IDF + сравнените топ слов по этой метрике с просто самыми популярными словами)